# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a Croissant AI-ready dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Explore the available record sets, field IDs, and schema structure.

Below, we list all top-level `RecordSet` entities (`@id` and `name`) in the dataset for reference. Each record set contains fields, each with their own `@id`.


In [ ]:
# Get all available record sets and their @ids
record_sets = list(dataset.record_sets)

print("Available Record Sets:")
for rs in record_sets:
    print(f"  - Name: {rs.name if hasattr(rs, 'name') else '<unnamed>'}, @id: {rs.id}")

# For each record set, print its field @ids and names
print("\nFields in each RecordSet:")
for rs in record_sets:
    print(f"\nRecordSet '{rs.id}':")
    for f in rs.fields:
        print(f"    - Field name: {getattr(f, 'name', '<unnamed>')}, @id: {f.id}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. All entities are referenced by their `@id`.

_Below, we'll extract data from each available record set, identified by its `@id`._


In [ ]:
# Build a list of record set @ids for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for RecordSet @id='{rs_id}': {df.shape[0]} rows, {df.shape[1]} columns.")

# For demonstration, choose the first available record set
if len(record_set_ids) > 0:
    primary_record_set_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for @id='{primary_record_set_id}':")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Typical EDA tasks can include filtering, normalization, and grouping. We will:
- Choose a numeric field by `@id` (if found), filter records above a threshold,
- Normalize that field, and
- Group by a categorical field if available.

If you know the field `@id` for a score column (e.g., GAD-7, PHQ-9), you can set it below. Otherwise, the notebook will attempt to use the first numeric field found.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

df = dataframes[primary_record_set_id]

# Find candidate numeric fields (@ids) from the record set
candidate_numeric_fields = []
for rs in record_sets:
    if rs.id == primary_record_set_id:
        for field in rs.fields:
            if hasattr(field, 'data_type') and getattr(field, 'data_type', None) in ['Integer', 'Float', 'Number', 'Double']:
                candidate_numeric_fields.append(field.id)

# Fall back to any numeric pandas columns if no type info present
if not candidate_numeric_fields:
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
    else:
        numeric_field_id = None
else:
    numeric_field_id = candidate_numeric_fields[0]

if numeric_field_id is not None:
    print(f"Selected numeric field @id for analysis: {numeric_field_id}\n")

    threshold = df[numeric_field_id].quantile(0.8) # 80th percentile as an example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Records with {numeric_field_id} > {threshold:.2f}")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a grouping field
    candidate_group_fields = [f.id for rs in record_sets if rs.id == primary_record_set_id for f in rs.fields if getattr(f, 'data_type', '') in ['Text', 'String'] or f.id != numeric_field_id]
    group_field_id = None
    for g in candidate_group_fields:
        if g in df.columns and df[g].nunique() > 1 and df[g].nunique() <= 10:
            group_field_id = g
            break
    if group_field_id:
        print(f"\nGrouping by {group_field_id}:")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped)
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and relationships with a grouping key (if found).

Below, we plot a histogram and, if a grouping field was found, a boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, showfliers=False)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
- The FAIR² Mental Health Survey Dataset from Kilifi County, Kenya was successfully loaded using its Croissant schema.
- Record sets, fields, and data rows were inspected by their `@id`, enabling reproducible data science.
- We explored numeric variables (e.g., GAD-7, PHQ-9, PSQ scores) and demonstrated filtering, normalization, and grouping.
- Visual exploration highlighted the data distributions and group-level variation.

This notebook provides a reproducible blueprint for AI-ready dataset processing!